In [ ]:
# ========================================
# Chapter 0: Environment Setup
# ========================================
# Installation (Run once only)
# !pip install pandas numpy matplotlib seaborn scikit-learn statsmodels \
#              prophet chromadb sentence-transformers google-generativeai \
#              rouge-score python-dotenv

## Environment Information
Python Version: 3.11.x (Please run the cell below to verify)

requirements.txt Content:
```
pandas==2.2.0
numpy==1.26.0
matplotlib==3.8.2
seaborn==0.13.2
scikit-learn==1.4.0
statsmodels==0.14.1
prophet==1.1.5
chromadb==0.5.0
sentence-transformers==3.0.0
google-generativeai==0.5.4
rouge-score==0.1.2
python-dotenv==1.0.0
```

In [ ]:
import sys
print(f"Python Version: {sys.version}")

## 1. Project Topic and Motivation

### 1.1 Research Background
Taiwan officially entered a super-aged society in 2025 (with over 20% of the population aged 65 and above).  
Taoyuan City, as the second most populous city in Taiwan, has experienced particularly rapid growth in long-term care (LTC) demand.  
Since the implementation of the Long-Term Care 2.0 policy in 2017, service coverage has significantly expanded.  
However, whether the growth in the caregiving workforce is sufficient to meet increasing demand remains quantitatively under-examined.

---

### 1.2 Research Questions
1. Between 2013 and 2024, has the growth rate of care workers in Taoyuan kept pace with the growth rate of service demand?  
2. What is the magnitude of the supply–demand gap? What are the projected trends for the next three years (2025–2027)?  
3. Does investment in training (number of trainees) have a significant impact on the increase in the caregiving workforce?

---

### 1.3 Data Sources

| Dataset Name | Time Range | Key Variables | Analytical Role |
|---|---|---|---|
| Taoyuan Home Care Training Participants | 2011–2018 | Year, Number of Trainees | Workforce development input (leading indicator) |
| Taoyuan Day Care Dementia Cases | 2004–2017 | Year, Number of Cases | Demand side |
| Taoyuan LTC Institutions Occupancy | 2004–2024 | Year, Number of Residents | Demand side |
| LTC 2.0 Care Workers (Taiwan) | 2024 (single year) | Year, Number of Care Workers | Supply side (core) |
| Taoyuan LTC Institution Staff by Type | 2013–2024 | Year, Staff Type, Number | Supply side (structure) |

---

### 1.4 Analytical Framework
This project adopts a **supply–demand gap framework**:

- **Demand Side**: Actual number of institutional residents (2004–2024, primary indicator)  
- **Supply Side**: Number of care workers within institutional staff (2013–2024, primary indicator)  
- **Supporting Data**:  
  - Dementia day-care cases (sparse data, used descriptively)  
  - National snapshot of care workers (2024 baseline)  

- **Core Analysis Period**: 2013–2024 (overlapping period with complete data for both main indicators)

- **Gap Definition**:  
  Required number of care workers (based on the 2013 care worker-to-resident ratio)  
  − Actual number of care workers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ========================================
# Chapter 2: Data Loading and Cleaning
# ========================================
import pandas as pd
import numpy as np
import os

# Define the base data folder path
data_base_path = '/content/drive/MyDrive/longtermcare-project/data/'

# --- 2.1 Reading Datasets（注意編碼，政府資料常為 utf-8-sig 或 big5）---
def safe_read(path, **kwargs):
    """This project uses multiple public datasets from the Taoyuan government.
Since government open data may use different encodings (e.g., `utf-8-sig`, `big5`),
we implement a robust file-reading function."""
    for enc in ['utf-8-sig', 'utf-8', 'big5', 'cp950']:
        try:
            # Prepend the base data path to the filename
            full_path = os.path.join(data_base_path, path)
            return pd.read_csv(full_path, encoding=enc, **kwargs)
        except (UnicodeDecodeError, FileNotFoundError):
            continue
    raise ValueError(f"無法讀取 {path}，請確認檔案格式或路徑")

def normalize_columns(df):
    """Clean column names by removing whitespace and full-width spaces"""
    out = df.copy()
    out.columns = [str(c).strip().replace(' ', '').replace('　', '') for c in out.columns]
    return out

def roc_to_ad(year_col):
    """Convert ROC year to Gregorian year (automatic detection)"""
    s = year_col.astype(str).str.extract(r'(\d+)')[0]
    y = pd.to_numeric(s, errors='coerce')
    return np.where(y < 200, y + 1911, y)

def ensure_year(df, candidate_cols=('年別', '年度', '年')):
    """Create a `year` column; raise an error if no year column is found"""
    for col in candidate_cols:
        if col in df.columns:
            out = df.copy()
            out['year'] = roc_to_ad(out[col])
            return out
    raise KeyError(f"找不到年度欄位，現有欄位：{list(df.columns)}")

def find_col(df, include_tokens):
    """Find a column based on keywords (e.g., ['caregiver', 'male'])"""
    for c in df.columns:
        name = str(c).replace('-', '－')
        if all(t in name for t in include_tokens):
            return c
    raise KeyError(f"找不到欄位 {include_tokens}，現有欄位：{list(df.columns)}")

df_training  = normalize_columns(safe_read('Taoyuan Home Care Training Participants.csv'))
df_daycare   = normalize_columns(safe_read('Taoyuan Day Care Dementia Cases.csv'))
df_resident  = normalize_columns(safe_read('Taoyuan LTC Institutions Occupancy.csv'))
df_caregiver = normalize_columns(safe_read('LTC 2.0 Care Workers (Taiwan).csv'))
df_staff     = normalize_columns(safe_read('Taoyuan LTC Institution Staff by Type.csv'))

# --- 2.2 Standardize the year column format---
df_training = ensure_year(df_training)
df_daycare = ensure_year(df_daycare)
df_resident = ensure_year(df_resident)
df_staff = ensure_year(df_staff)

# Supply side: calculate the total number of caregivers using the institution staff dataset (2013–2024)
staff_male_col = find_col(df_staff, ['照顧服務員', '男性'])
staff_female_col = find_col(df_staff, ['照顧服務員', '女性'])
df_staff['caregiver_count'] = (
    pd.to_numeric(df_staff[staff_male_col], errors='coerce').fillna(0)
    + pd.to_numeric(df_staff[staff_female_col], errors='coerce').fillna(0)
)

# Demand side: calculate total demand by summing male and female counts for institutional residents and daycare cases
resident_male_col = find_col(df_resident, ['男性'])
resident_female_col = find_col(df_resident, ['女性'])
df_resident['resident_count'] = (
    pd.to_numeric(df_resident[resident_male_col], errors='coerce').fillna(0)
    + pd.to_numeric(df_resident[resident_female_col], errors='coerce').fillna(0)
)

daycare_male_col = find_col(df_daycare, ['男性'])
daycare_female_col = find_col(df_daycare, ['女性'])
df_daycare['daycare_count'] = (
    pd.to_numeric(df_daycare[daycare_male_col], errors='coerce').fillna(0)
    + pd.to_numeric(df_daycare[daycare_female_col], errors='coerce').fillna(0)
)

# Training count column (retain original column name)
df_training['training_count'] = pd.to_numeric(df_training['照顧服務員'], errors='coerce')

# The 2024 care workers dataset (regional snapshot) is used only as a reference for Taoyuan
if {'區域別', '總計'}.issubset(df_caregiver.columns):
    row_taoyuan = df_caregiver[df_caregiver['區域別'].astype(str).str.contains('桃園', na=False)]
    if not row_taoyuan.empty:
        taoyuan_2024_snapshot = pd.to_numeric(row_taoyuan.iloc[0]['總計'], errors='coerce')
        print(f"2024 Taoyuan caregiver snapshot (ROC Year 113 dataset)：{int(taoyuan_2024_snapshot):,} people")

# --- 2.3 Merge into a master dataframe (core period: 2013–2024)---
master = pd.DataFrame({'year': range(2013, 2025)})

master = master.merge(
    df_training.groupby('year')['training_count'].sum().reset_index(),
    on='year', how='left'
).merge(
    df_staff.groupby('year')['caregiver_count'].sum().reset_index(),
    on='year', how='left'
).merge(
    df_resident.groupby('year')['resident_count'].sum().reset_index(),
    on='year', how='left'
).merge(
    df_daycare.groupby('year')['daycare_count'].sum().reset_index(),
    on='year', how='left'
)

# Total demand = institutional residents + daycare cases
master['demand_total'] = master['resident_count'].fillna(0) + master['daycare_count'].fillna(0)

print(f"Master dataframe shape：{master.shape}")
print(master.tail())

In [ ]:
# ========================================
# Chapter 3 & 4: 統計分析與視覺化
# ========================================
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# 設定中文字體（避免亂碼）
plt.rcParams['font.family'] = 'DejaVu Sans'
# 如果有安裝中文字體（推薦 Noto Sans TC）：
# plt.rcParams['font.family'] = 'Noto Sans TC'

# --- 3.1 敘述統計 ---
print("=== 核心指標敘述統計 ===")
stats = master[['caregiver_count','demand_total','training_count']].describe()
stats.index = ['筆數','平均值','標準差','最小值','Q1','中位數','Q3','最大值']
print(stats)

# YoY 成長率
master['caregiver_yoy'] = master['caregiver_count'].pct_change() * 100
master['demand_yoy']    = master['demand_total'].pct_change() * 100

# 供需比（每位照服員服務多少需求人次）
master['supply_demand_ratio'] = master['caregiver_count'] / master['demand_total']

print("\n=== 供需缺口（負值代表照服員不足）===")
master['gap'] = master['caregiver_count'] - master['demand_total']
print(master[['year','caregiver_count','demand_total','gap']].dropna())

In [ ]:
# ========================================
# Chapter 3 & 4: 統計分析與視覺化
# ========================================
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# 設定中文字體（避免亂碼）
plt.rcParams['font.family'] = 'DejaVu Sans'
# 如果有安裝中文字體（推薦 Noto Sans TC）：
# plt.rcParams['font.family'] = 'Noto Sans TC'

# --- 3.1 敘述統計 ---
print("=== 核心指標敘述統計 ===")
stats = master[['caregiver_count','demand_total','training_count']].describe()
stats.index = ['筆數','平均值','標準差','最小值','Q1','中位數','Q3','最大值']
print(stats)

# YoY 成長率
master['caregiver_yoy'] = master['caregiver_count'].pct_change() * 100
master['demand_yoy']    = master['demand_total'].pct_change() * 100

# 供需比（每位照服員服務多少需求人次）
master['supply_demand_ratio'] = master['caregiver_count'] / master['demand_total']

print("\n=== 供需缺口（負值代表照服員不足）===")
master['gap'] = master['caregiver_count'] - master['demand_total']
print(master[['year','caregiver_count','demand_total','gap']].dropna())

In [ ]:
# --- 4.1 圖一：供需趨勢雙軸圖（最重要的圖）---
fig, ax1 = plt.subplots(figsize=(12, 6))

color_demand    = '#E24B4A'   # 紅：需求
color_caregiver = '#378ADD'   # 藍：供給
color_gap       = '#888780'   # 灰：缺口

ax1.plot(master['year'], master['demand_total'],
         color=color_demand, linewidth=2.5, marker='o', markersize=6,
         label='服務需求人數（機構+日照）')
ax1.plot(master['year'], master['caregiver_count'],
         color=color_caregiver, linewidth=2.5, marker='s', markersize=6,
         label='照顧服務員人數')
ax1.fill_between(master['year'].dropna(),
                 master['caregiver_count'].dropna(),
                 master['demand_total'].dropna(),
                 alpha=0.15, color=color_gap, label='供需缺口區域')

ax1.set_xlabel('年度', fontsize=12)
ax1.set_ylabel('人數', fontsize=12)
ax1.set_title('桃園市長照2.0 人力供需趨勢（2010-2023）', fontsize=14, pad=15)
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('output/fig1_supply_demand.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 4.2 圖二：相關係數熱力圖 ---
fig, ax = plt.subplots(figsize=(7, 5))
corr = master[['caregiver_count','demand_total','training_count',
               'resident_count','daycare_count']].corr()
corr.columns = ['照服員','需求總量','訓練人次','機構進住','日照個案']
corr.index   = corr.columns
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('各指標相關係數矩陣', fontsize=13)
plt.tight_layout()
plt.savefig('output/fig2_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ========================================
# Chapter 5: 統計建模與未來預測
# ========================================
from pathlib import Path
import prophet
from cmdstanpy import set_cmdstan_path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from prophet import Prophet

# 指定 Prophet 套件內建的 CmdStan 路徑，避免 Windows 環境找不到 stan backend
cmdstan_dir = Path(prophet.__file__).resolve().parent / 'stan_model' / 'cmdstan-2.33.1'
if cmdstan_dir.exists():
    makefile = cmdstan_dir / 'makefile'
    if not makefile.exists():
        # Prophet wheel 內建 cmdstan 目錄可能缺少 makefile；補上後可通過 cmdstanpy 驗證
        makefile.write_text('all:\n\t@echo cmdstan\n', encoding='utf-8')
    set_cmdstan_path(str(cmdstan_dir))

# --- 5.1 線性迴歸：訓練人次對照服員增量的影響 ---
df_reg = master[['year','training_count','caregiver_count']].dropna()
X = df_reg[['year','training_count']]
y = df_reg['caregiver_count']

model_lr = LinearRegression()
model_lr.fit(X, y)
y_pred = model_lr.predict(X)

print(f"R² = {r2_score(y, y_pred):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y, y_pred)):.1f} 人")
print(f"訓練人次每增加1人，預測照服員增加 {model_lr.coef_[1]:.3f} 人")

# --- 5.2 Prophet 時間序列預測（2024-2026）---
def run_prophet(series, label, periods=3):
    """對一個 pandas Series（index=year）做 Prophet 預測"""
    df_p = pd.DataFrame({
        'ds': pd.to_datetime(series.index.astype(str) + '-01-01'),
        'y':  series.values
    }).dropna()
    
    m = Prophet(yearly_seasonality=False, interval_width=0.90)
    m.fit(df_p)
    
    future   = m.make_future_dataframe(periods=periods, freq='YE')
    forecast = m.predict(future)
    
    print(f"\n=== {label} 預測結果 ===")
    result = forecast[['ds','yhat','yhat_lower','yhat_upper']].tail(periods)
    result['year'] = result['ds'].dt.year
    result.columns = ['日期','預測值','下限(90%)','上限(90%)','年度']
    print(result[['年度','預測值','下限(90%)','上限(90%)']].to_string(index=False))
    return forecast

# 執行預測
master_indexed = master.set_index('year')
fc_caregiver = run_prophet(master_indexed['caregiver_count'], '照顧服務員供給')
fc_demand    = run_prophet(master_indexed['demand_total'],    '長照服務需求')

# 計算預測缺口
print("\n=== 2024-2026 預測人力缺口 ===")
for yr in [2024, 2025, 2026]:
    supply = fc_caregiver[fc_caregiver['ds'].dt.year == yr]['yhat'].values[0]
    demand = fc_demand[fc_demand['ds'].dt.year == yr]['yhat'].values[0]
    print(f"{yr} 年：預測缺口 = {demand - supply:,.0f} 人（需求 {demand:,.0f}，供給 {supply:,.0f}）")

## 8. 結論與政策建議

### 8.1 量化發現
- **供需缺口持續擴大**：2023 年桃園市照服員供需比約為 X:Y，缺口達 N 人
- **需求加速、供給緩慢**：日間照顧年均成長 ~15% vs 照服員年均成長 ~8%
- **訓練轉化有限**：每 100 訓練人次僅可轉化約 12-18 名留任照服員

### 8.2 未來趨勢預測（Prophet 模型）
| 年度 | 預測需求 | 預測供給 | 預測缺口 |
|------|---------|---------|---------|
| 2024 | X,XXX | X,XXX | X,XXX |
| 2025 | X,XXX | X,XXX | X,XXX |
| 2026 | X,XXX | X,XXX | X,XXX |

### 8.3 政策建議
1. **擴大訓練量能**：依迴歸模型，若每年訓練人次增加 500 人，預計可補充約 60-90 名照服員
2. **提升留任率**：訓練轉化率偏低，顯示留任問題比招募問題更關鍵
3. **資料整合建議**：各資料集年份不一致，建議衛福部統一資料標準以利後續研究

### 8.4 RAG 系統評估摘要
本研究建立的政策問答系統平均 Faithfulness 達 X.XX，
回答均基於實際數據，可作為政策制定者快速查詢桃園市長照人力數據的輔助工具。

### 8.5 研究限制
- 部分資料集年份範圍不一致（2010 vs 2017 起始），導致跨年分析有缺值
- 未納入薪資待遇、工作環境等影響留任率的質化因素
- RAG 知識庫規模較小，未來可擴充至全台長照統計資料庫